# Solar Filament Segmentation Challenge 2026 — Final Kaggle Pipeline

**Goal:** a real, reproducible model-generated submission with held-out evaluation and instance-aware post-processing.

This notebook intentionally does **not** install or downgrade PyTorch/Torchvision. The earlier notebook logged validation Dice near zero and produced test masks from a hard-coded payload; those predictions are not used here. Every submitted mask in this notebook is created by inference on the attached test images.

Official metric guidance: [competition overview](https://www.kaggle.com/competitions/filament-segmentation-2026/overview/final-submission) · [organizer self-evaluation notebook](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook).

**First run:** set Kaggle Accelerator to GPU. This notebook's `RUN_MODE = "competition"` does a smoke update, trains a U-Net baseline, trains a YOLO11-seg proposal model, tunes detector-guided fusion on tuning views, reports a separate holdout, retrains selected models on all labelled annotation views, and writes `/kaggle/working/submission.csv`. Switch to `"smoke"` only to verify the environment quickly; smoke-mode output is not intended as a competition entry.

In [ ]:
# ============================================================
# CELL 1 — GITHUB SOURCE BOOTSTRAP
# ============================================================
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
import py_compile

REPO_URL = "https://github.com/ak11pro/Chatgpt.web.git"
REPO_BRANCH = "codex/audit-provided-notebook-for-weaknesses"
REPO_DIR = Path("/kaggle/working/Chatgpt.web")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)

PIPELINE_PATH = REPO_DIR / "solar_filament" / "pipeline.py"
assert PIPELINE_PATH.is_file(), f"Pipeline source missing: {PIPELINE_PATH}"
sys.path.insert(0, str(REPO_DIR))

REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

print("Repository:", REPO_DIR)
print("Branch:", REPO_BRANCH)
print("Commit:", REPO_COMMIT)
print("Pipeline:", PIPELINE_PATH)
py_compile.compile(str(PIPELINE_PATH), doraise=True)
print("Pipeline Python syntax: PASS")


In [ ]:
# ============================================================
# CELL 2 — INSTALL ONLY MISSING NON-TORCH DEPENDENCIES
# Do not upgrade/downgrade torch, torchvision, or numpy here.
# ============================================================
import importlib.util

PACKAGE_FOR_MODULE = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "cv2": "opencv-python-headless",
    "scipy": "scipy",
    "skimage": "scikit-image",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "pycocotools": "pycocotools",
}

missing_packages = [
    package for module, package in PACKAGE_FOR_MODULE.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    print("Installing missing packages only:", missing_packages)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_packages],
        check=True,
    )
else:
    print("All non-Torch dependencies already exist.")


In [ ]:
# ============================================================
# CELL 3 — SAFE TORCH/CUDA PREFLIGHT
# ============================================================
try:
    import torch
except Exception as exc:
    print("PyTorch itself failed to import.")
    print(type(exc).__name__, str(exc))
    subprocess.run(
        [sys.executable, "-m", "pip", "show", "torch", "torchvision"],
        check=False,
    )
    raise RuntimeError(
        "Do not reinstall a random Torch wheel. Restart the Kaggle session, "
        "remove any earlier torch/torchvision pip-install cell, and run this "
        "notebook in a fresh Kaggle GPU session."
    ) from exc

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

TORCHVISION_OK = False
TORCHVISION_ERROR = None
try:
    import torchvision
    from torchvision.models import ResNet34_Weights, resnet34
    TORCHVISION_OK = True
    print("Torchvision:", torchvision.__version__)
    print("Torchvision ResNet-34: available")
except Exception as exc:
    TORCHVISION_ERROR = f"{type(exc).__name__}: {exc}"
    print("Torchvision import failed:", TORCHVISION_ERROR)
    print("The pipeline has a pure-PyTorch ResNet-shaped fallback. It will not have ImageNet weights.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: CUDA is unavailable. Competition mode requires a GPU.")

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# CELL 4 — IMPORT PIPELINE + RUN SETTINGS
# ============================================================
import gc
import hashlib
import json
import logging
import random
from dataclasses import asdict
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import cv2
import itertools
import re
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import DataLoader

from solar_filament.pipeline import (
    PipelineConfig,
    FilamentSegmentationModel,
    SolarFilamentDataset,
    append_experiment,
    build_grad_scaler,
    build_instance_masks,
    build_semantic_mask,
    cache_validation_predictions,
    compute_pq,
    compute_segmentation_metrics,
    compound_loss,
    create_submission,
    environment_report,
    find_competition_data,
    group_train_validation_split,
    load_coco_annotations,
    measure_foreground_prevalence,
    optimize_postprocessing,
    run_synthetic_metric_tests,
    set_seed,
    train_one_epoch,
    validate,
    reconstruct_instances,
    predict_tiled,
    detect_solar_disk,
    mask_to_rle,
    validate_rle,
    list_image_files,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Use "smoke" for a quick compatibility check; use "competition" for the full run.
RUN_MODE = "competition"
FINAL_RETRAIN_ALL_DATA = True

SEED = 42
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_TAG += "_" + REPO_COMMIT[:7]

if RUN_MODE not in {"smoke", "competition"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'competition'.")
if RUN_MODE == "competition" and not torch.cuda.is_available():
    raise RuntimeError("Set Kaggle Accelerator = GPU before running competition mode.")
if RUN_MODE == "competition" and not TORCHVISION_OK:
    raise RuntimeError(
        "Competition mode includes YOLO instance proposals and requires a working torchvision. "
        "Restart a fresh Kaggle GPU session; remove any earlier torch/torchvision pip-install cells. "
        f"Current torchvision error: {TORCHVISION_ERROR}"
    )

config = PipelineConfig(
    seed=SEED,
    image_size=2048,
    train_tile_size=512 if RUN_MODE == "smoke" else 768,
    tile_overlap=0.35,
    inference_batch_size=1 if RUN_MODE == "smoke" else 2,
    in_channels=1,
    pretrained_encoder=True,
    epochs=1 if RUN_MODE == "smoke" else 18,
    batch_size=1,
    gradient_accumulation=2,
    learning_rate=2e-4,
    weight_decay=1e-4,
    num_workers=2,
    use_amp=True,
    val_fraction=0.20,
    threshold=0.40,
    min_component_area=200,
    watershed=True,
    watershed_peak_distance=18,
    boundary_mode="guidance",
    boundary_head=True,
    use_tta=False,
    use_solar_disk_mask=True,  # tested both on/off by PQ-driven post-processing search
    max_train_images=None,
    samples_per_epoch=32 if RUN_MODE == "smoke" else 900,
    mask_cache_dir="/kaggle/working/solar_filament_mask_cache_v2",
)

set_seed(SEED)
torch.set_num_threads(max(1, min(8, os.cpu_count() or 2)))

print(json.dumps(environment_report(config), indent=2))
print("Run mode:", RUN_MODE)
print("Run tag:", RUN_TAG)
run_synthetic_metric_tests()
print("Synthetic PQ tests passed.")


In [ ]:
# ============================================================
# CELL 5 — FIND DATA + SPLIT BY PHYSICAL OBSERVATION
# Every annotation view of the same JPEG stays in one split.
# ============================================================
data = find_competition_data("/kaggle/input")
index = load_coco_annotations(data.annotation_json)

train_ids, val_ids = group_train_validation_split(
    index, config.val_fraction, config.seed
)

physical_train = {
    Path(index.images[i]["file_name"]).name for i in train_ids
}
physical_val = {
    Path(index.images[i]["file_name"]).name for i in val_ids
}
assert physical_train.isdisjoint(physical_val), "Physical observation leakage detected."

val_stems = sorted(physical_val)
rng = np.random.default_rng(SEED)

# Keep entire calendar dates together in the tune/holdout split as well. Solar
# filaments can persist across consecutive frames, so a random frame split
# would make the final validation estimate too optimistic.
val_dates = sorted({stem[:8] for stem in val_stems})
val_dates_array = np.asarray(val_dates, dtype=object)
rng.shuffle(val_dates_array)

cut = max(1, len(val_dates_array) // 2)
tune_date_pool = set(val_dates_array[:cut].tolist())
holdout_dates = set(val_dates_array[cut:].tolist())
tune_pool = [
    stem for stem in val_stems
    if stem[:8] in tune_date_pool
]
holdout_stems = {
    stem for stem in val_stems
    if stem[:8] in holdout_dates
}

# Limit expensive post-processing sweeps to a reproducible subset of tuning
# observations. The holdout dates remain untouched for the final reported score.
n_tune = min(24 if RUN_MODE == "competition" else 4, len(tune_pool))
tune_stems = set(rng.choice(tune_pool, size=n_tune, replace=False).tolist())

tune_ids = [
    i for i in val_ids
    if Path(index.images[i]["file_name"]).name in tune_stems
]
holdout_ids = [
    i for i in val_ids
    if Path(index.images[i]["file_name"]).name in holdout_stems
]

quick_stems = set(rng.choice(
    sorted(tune_stems),
    size=min(12 if RUN_MODE == "competition" else 2, len(tune_stems)),
    replace=False,
).tolist())
quick_val_ids = [
    i for i in tune_ids
    if Path(index.images[i]["file_name"]).name in quick_stems
]

assert tune_ids and holdout_ids and quick_val_ids
assert {Path(index.images[i]["file_name"]).name for i in tune_ids}.isdisjoint(
    {Path(index.images[i]["file_name"]).name for i in holdout_ids}
)
assert {Path(index.images[i]["file_name"]).name[:8] for i in tune_ids}.isdisjoint(
    {Path(index.images[i]["file_name"]).name[:8] for i in holdout_ids}
)

split_summary = {
    "train_annotation_entries": len(train_ids),
    "validation_annotation_entries": len(val_ids),
    "training_physical_images": len(physical_train),
    "validation_physical_images": len(physical_val),
    "postprocess_tuning_physical_images": len(tune_stems),
    "final_holdout_physical_images": len(holdout_stems),
    "quick_validation_physical_images": len(quick_stems),
    "tuning_dates": len(tune_date_pool),
    "final_holdout_dates": len(holdout_dates),
    "train_instance_count": sum(len(index.annotations_by_image.get(i, [])) for i in train_ids),
}
print(json.dumps(split_summary, indent=2))


In [ ]:
# ============================================================
# CELL 6 — VISUAL ANNOTATION SANITY CHECK
# Inspect this grid. Stop if polygons are empty or misaligned.
# ============================================================
vis_ids = rng.choice(
    train_ids,
    size=min(12, len(train_ids)),
    replace=False,
)
fig, axes = plt.subplots(3, 4, figsize=(18, 13))

for ax, image_id in zip(axes.flat, vis_ids):
    meta = index.images[image_id]
    img_path = data.train_images / meta["file_name"]
    with Image.open(img_path) as handle:
        image = np.asarray(handle.convert("L"))
    instances = build_instance_masks(index, image_id)
    semantic = build_semantic_mask(index, image_id)

    assert image.shape == semantic.shape == (2048, 2048)
    assert all(m.shape == image.shape and m.any() for m in instances)

    ax.imshow(image, cmap="gray")
    ax.imshow(semantic, cmap="Reds", alpha=0.40)
    ax.set_title(f"{Path(meta['file_name']).name}\nviews={len(instances)}")
    ax.axis("off")

for ax in axes.flat[len(vis_ids):]:
    ax.axis("off")

fig.tight_layout()
sanity_path = WORKING / f"annotation_sanity_{RUN_TAG}.png"
fig.savefig(sanity_path, dpi=130)
plt.show()
print("Saved annotation sanity grid:", sanity_path)


In [ ]:
# ============================================================
# CELL 7 — BUILD CACHED DATASETS + QUICK VALIDATION SUBSET
# Masks are rasterized once into a compressed, versioned cache.
# ============================================================
print("Building training dataset and crop candidates...")
train_dataset = SolarFilamentDataset(
    index, data.train_images, train_ids, config, training=True
)
quick_val_dataset = SolarFilamentDataset(
    index, data.train_images, quick_val_ids, config, training=False
)
tune_dataset = SolarFilamentDataset(
    index, data.train_images, tune_ids, config, training=False
)
holdout_dataset = SolarFilamentDataset(
    index, data.train_images, holdout_ids, config, training=False
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(config.num_workers > 0),
)

print("Device:", device)
print("Training crop samples per epoch:", len(train_dataset))
print("Quick validation views:", len(quick_val_dataset))
print("Tuning views:", len(tune_dataset))
print("Independent holdout views:", len(holdout_dataset))


In [ ]:
# ============================================================
# CELL 8 — GPU FORWARD/BACKWARD SMOKE TEST
# Catch import, shape, loss, and CUDA issues before long training.
# ============================================================
model = FilamentSegmentationModel(config).to(device)
print("ImageNet-pretrained encoder loaded:", getattr(model, "pretrained_loaded", False))
if RUN_MODE == "competition" and not getattr(model, "pretrained_loaded", False):
    raise RuntimeError(
        "The pretrained ResNet-34 weights were unavailable. Check Kaggle Internet access and restart "
        "before training. Refusing a long competition run with randomly initialized encoder weights."
    )

batch = next(iter(train_loader))
images = batch["image"].to(device)
semantic = batch["semantic"].to(device)
boundary = batch["boundary"].to(device)

model.train()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

optimizer.zero_grad(set_to_none=True)
outputs = model(images)
loss = compound_loss(
    outputs,
    {"semantic": semantic, "boundary": boundary},
    config,
    pos_weight=20.0,
)
assert torch.isfinite(loss).item(), f"Non-finite smoke-test loss: {loss.item()}"
loss.backward()
optimizer.step()

print({
    "input_shape": tuple(images.shape),
    "semantic_output_shape": tuple(outputs["semantic"].shape),
    "boundary_output_shape": tuple(outputs["boundary"].shape),
    "smoke_loss": float(loss.detach().cpu()),
    "finite_gradients": all(
        p.grad is None or torch.isfinite(p.grad).all().item()
        for p in model.parameters()
    ),
})
del batch, images, semantic, boundary, outputs
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# CELL 9 — TRAIN AND SELECT CHECKPOINT ON QUICK HELD-OUT PQ
# ============================================================
# This prevalence sample is intentionally bounded to keep startup time sane.
prevalence = measure_foreground_prevalence(
    index, train_ids[:min(48, len(train_ids))]
)
pos_weight = float(np.clip(prevalence["background_to_foreground"], 1.0, 50.0))
print("Sampled prevalence:", prevalence)
print("BCE pos_weight:", pos_weight)

# Reinitialize after the smoke update so the measured run begins cleanly.
model = FilamentSegmentationModel(config).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config.epochs
)
scaler = build_grad_scaler(device, config.use_amp)

best_path = WORKING / f"best_model_{RUN_TAG}.pth"
history = []
best_metric = -1.0
best_epoch = 0
epochs_without_improvement = 0
patience = 5
started = time.perf_counter()

for epoch in range(config.epochs):
    epoch_started = time.perf_counter()

    train_loss = train_one_epoch(
        model, train_loader, optimizer, scaler,
        config, device, pos_weight
    )
    scheduler.step()

    quick_metrics = validate(
        model, quick_val_dataset, config, device
    )
    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "epoch_seconds": time.perf_counter() - epoch_started,
        **quick_metrics,
    }
    history.append(row)
    print(json.dumps(row, indent=2, default=float))

    if quick_metrics["global_pq"] > best_metric + 1e-6:
        best_metric = quick_metrics["global_pq"]
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        torch.save({
            "model": model.state_dict(),
            "config": asdict(config),
            "metrics": quick_metrics,
            "epoch": best_epoch,
            "pos_weight": pos_weight,
            "repo_commit": REPO_COMMIT,
        }, best_path)
        print("Saved improved checkpoint:", best_path)
    else:
        epochs_without_improvement += 1

    if RUN_MODE == "competition" and epochs_without_improvement >= patience:
        print("Early stopping: quick validation PQ has not improved.")
        break

pd.DataFrame(history).to_csv(WORKING / f"training_history_{RUN_TAG}.csv", index=False)
if not best_path.exists():
    raise RuntimeError("No checkpoint was saved; refusing untrained inference.")

checkpoint = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model"])
print("Selected epoch:", best_epoch)
print("Selected quick global PQ:", best_metric)
print("Training seconds:", round(time.perf_counter() - started, 1))


In [ ]:
# ============================================================
# CELL 10 — TUNE POST-PROCESSING ON TUNING VIEWS ONLY
# The final holdout is not used by this search.
# ============================================================
tune_cache_dir = WORKING / f"tune_cache_{RUN_TAG}"
tune_cache = cache_validation_predictions(
    model, tune_dataset, config, device, tune_cache_dir
)

optimized_config, search_table = optimize_postprocessing(
    [entry[0] for entry in tune_cache],
    [entry[1] for entry in tune_cache],
    [entry[2] for entry in tune_cache],
    config,
    disk_masks=[entry[3] for entry in tune_cache],
)

search_path = WORKING / f"postprocessing_search_{RUN_TAG}.csv"
search_table.to_csv(search_path, index=False)

print("Best post-processing candidates:")
display(search_table.head(12))
print("Selected configuration:")
print(json.dumps(asdict(optimized_config), indent=2))


In [ ]:
# ============================================================
# CELL 11 — FINAL INDEPENDENT HELD-OUT EVALUATION
# Do not make further tuning decisions from these results.
# ============================================================
holdout_started = time.perf_counter()
holdout_metrics = validate(
    model,
    holdout_dataset,
    optimized_config,
    device,
)
holdout_seconds = time.perf_counter() - holdout_started

print("INDEPENDENT HELD-OUT METRICS (measured)")
print(json.dumps(holdout_metrics, indent=2))
print("Holdout evaluation seconds:", round(holdout_seconds, 1))

with (WORKING / f"holdout_metrics_{RUN_TAG}.json").open("w") as handle:
    json.dump(holdout_metrics, handle, indent=2)

with (WORKING / f"optimized_config_{RUN_TAG}.json").open("w") as handle:
    json.dump(asdict(optimized_config), handle, indent=2)

append_experiment(
    WORKING / "experiments.csv",
    f"resnet34_unet_boundary_{RUN_TAG}",
    optimized_config,
    holdout_metrics,
    time.perf_counter() - started,
    pos_weight,
    split_summary,
)


In [ ]:
# ============================================================
# CELL 12 — TRAIN YOLO11-SEG INSTANCE PROPOSALS
# Training labels are converted from the permitted COCO training polygons.
# All annotation views for the same physical JPEG remain in the same split.
# ============================================================
USE_DETECTOR_FUSION = False
fusion_best_params = None
fusion_holdout_metrics = None
yolo_best_path = None
best_yolo_epoch = None
fusion_search_table = pd.DataFrame()

if RUN_MODE == "competition":
    if not TORCHVISION_OK:
        raise RuntimeError("YOLO needs a working torchvision; fix the Kaggle environment before training.")

    # Install the Python wrapper without allowing pip to replace Torch/CUDA.
    if importlib.util.find_spec("ultralytics") is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "ultralytics==8.4.173"],
            check=True,
        )
    if importlib.util.find_spec("ultralytics_thop") is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "ultralytics-thop"],
            check=False,
        )
    from ultralytics import YOLO
    import ultralytics
    print("Ultralytics:", ultralytics.__version__)
    print("Torch remains:", torch.__version__, "| torchvision:", torchvision.__version__)

    # Make a small independent subset of the held-out partition for detector-fusion evaluation.
    fusion_holdout_stems = sorted(
        rng.choice(
            sorted(holdout_stems),
            size=min(16, len(holdout_stems)),
            replace=False,
        ).tolist()
    )
    fusion_holdout_stem_set = set(fusion_holdout_stems)
    fusion_holdout_ids = [
        image_id for image_id in holdout_ids
        if Path(index.images[image_id]["file_name"]).stem in fusion_holdout_stem_set
    ]
    assert fusion_holdout_ids, "No independent fusion holdout was selected."

    fusion_holdout_dataset = SolarFilamentDataset(
        index, data.train_images, fusion_holdout_ids, config, training=False
    )
    fusion_holdout_cache = cache_validation_predictions(
        model,
        fusion_holdout_dataset,
        config,
        device,
        WORKING / f"fusion_holdout_cache_{RUN_TAG}",
    )

    YOLO_DATA_ROOT = WORKING / f"yolo_data_{RUN_TAG}"
    YOLO_RUN_ROOT = WORKING / f"yolo_runs_{RUN_TAG}"
    if YOLO_DATA_ROOT.exists():
        shutil.rmtree(YOLO_DATA_ROOT)

    def _safe_yolo_image_name(image_id, file_name):
        token = hashlib.sha1(str(image_id).encode("utf-8")).hexdigest()[:10]
        path = Path(file_name)
        suffix = path.suffix.lower() or ".jpg"
        return f"{path.stem}_{token}{suffix}"

    def _polygon_area_xy(points):
        points = np.asarray(points, dtype=np.float32).reshape(-1, 2)
        if len(points) < 3:
            return 0.0
        return float(
            0.5 * abs(
                np.dot(points[:, 0], np.roll(points[:, 1], 1))
                - np.dot(points[:, 1], np.roll(points[:, 0], 1))
            )
        )

    def export_yolo_views(image_ids, split_name):
        image_dir = YOLO_DATA_ROOT / "images" / split_name
        label_dir = YOLO_DATA_ROOT / "labels" / split_name
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)
        multi_polygon_annotations = 0
        valid_polygons = 0

        for image_id in image_ids:
            meta = index.images[image_id]
            source = data.train_images / meta["file_name"]
            if not source.is_file():
                raise FileNotFoundError(f"Missing training image: {source}")
            yolo_name = _safe_yolo_image_name(image_id, meta["file_name"])
            destination = image_dir / yolo_name

            if not destination.exists():
                try:
                    destination.symlink_to(source.resolve())
                except OSError:
                    shutil.copy2(source, destination)

            rows = []
            for record in index.annotations_by_image.get(image_id, []):
                segmentation = record.segmentation
                if not isinstance(segmentation, list) or not segmentation:
                    continue
                polygons = [
                    poly for poly in segmentation
                    if isinstance(poly, (list, tuple)) and len(poly) >= 6 and len(poly) % 2 == 0
                ]
                if not polygons:
                    continue
                if len(polygons) > 1:
                    multi_polygon_annotations += 1
                # The supplied challenge schema describes one polygon per filament.
                # If an anomalous annotation has more than one, retain the largest
                # contour rather than concatenating separate contours incorrectly.
                polygon = max(polygons, key=_polygon_area_xy)
                points = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)
                if len(points) < 3:
                    continue
                points[:, 0] = np.clip(points[:, 0] / float(meta["width"]), 0.0, 1.0)
                points[:, 1] = np.clip(points[:, 1] / float(meta["height"]), 0.0, 1.0)
                if not np.isfinite(points).all():
                    continue
                coordinates = " ".join(f"{value:.6f}" for value in points.reshape(-1))
                rows.append("0 " + coordinates)
                valid_polygons += 1

            (label_dir / f"{Path(yolo_name).stem}.txt").write_text(
                "\n".join(rows), encoding="utf-8"
            )

        return {
            "annotation_views": len(image_ids),
            "images_written": len(list(image_dir.iterdir())),
            "labels_written": len(list(label_dir.iterdir())),
            "valid_filament_polygons": valid_polygons,
            "multi_polygon_records_used_largest_contour": multi_polygon_annotations,
        }

    train_yolo_stats = export_yolo_views(train_ids, "train")
    val_yolo_stats = export_yolo_views(tune_ids, "val")

    assert set(
        Path(index.images[i]["file_name"]).name for i in train_ids
    ).isdisjoint(
        set(Path(index.images[i]["file_name"]).name for i in tune_ids)
    ), "YOLO train/validation has physical-image leakage."

    YOLO_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    yolo_yaml = YOLO_DATA_ROOT / "dataset.yaml"
    yolo_yaml.write_text(
        f"path: {YOLO_DATA_ROOT}\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n"
        "  0: filament\n",
        encoding="utf-8",
    )

    print("YOLO train export:", json.dumps(train_yolo_stats, indent=2))
    print("YOLO validation export:", json.dumps(val_yolo_stats, indent=2))
    print("YOLO dataset YAML:", yolo_yaml)

    # Release the U-Net's training optimizer/loader GPU state before detector training.
    model.to("cpu")
    for _name in (
        "optimizer", "scheduler", "scaler", "train_loader",
        "train_dataset", "quick_val_dataset", "tune_dataset", "holdout_dataset",
    ):
        globals().pop(_name, None)
    gc.collect()
    torch.cuda.empty_cache()

    YOLO_EPOCHS = 1 if RUN_MODE == "smoke" else 30
    yolo_model = YOLO("yolo11s-seg.pt")
    split_yolo_run_name = f"split_{RUN_TAG}"
    yolo_output_dir = YOLO_RUN_ROOT / split_yolo_run_name

    yolo_model.train(
        data=str(yolo_yaml),
        imgsz=1024,
        epochs=YOLO_EPOCHS,
        batch=2,
        device=0,
        workers=2,
        patience=7,
        seed=SEED,
        deterministic=False,
        single_cls=True,
        cos_lr=True,
        plots=False,
        cache=False,
        val=True,
        mosaic=0.0,
        close_mosaic=0,
        mixup=0.0,
        copy_paste=0.0,
        fliplr=0.5,
        flipud=0.5,
        degrees=0.0,
        shear=0.0,
        perspective=0.0,
        translate=0.03,
        scale=0.10,
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.10,
        overlap_mask=True,
        mask_ratio=2,
        project=str(YOLO_RUN_ROOT),
        name=split_yolo_run_name,
        exist_ok=True,
    )

    yolo_best_path = yolo_output_dir / "weights" / "best.pt"
    if not yolo_best_path.is_file():
        raise FileNotFoundError(f"YOLO training did not produce best.pt: {yolo_best_path}")

    best_yolo_epoch = YOLO_EPOCHS
    results_csv = yolo_output_dir / "results.csv"
    if results_csv.is_file():
        yolo_history = pd.read_csv(results_csv)
        mask_metric_columns = [
            column for column in yolo_history.columns
            if "mAP50-95(M)" in column
        ]
        if mask_metric_columns and not yolo_history.empty:
            values = pd.to_numeric(yolo_history[mask_metric_columns[0]], errors="coerce")
            if values.notna().any():
                best_row = int(values.idxmax())
                if "epoch" in yolo_history.columns:
                    epoch_value = pd.to_numeric(
                        pd.Series([yolo_history.loc[best_row, "epoch"]]),
                        errors="coerce",
                    ).iloc[0]
                    if np.isfinite(epoch_value):
                        best_yolo_epoch = int(max(1, round(float(epoch_value))))
                    else:
                        best_yolo_epoch = best_row + 1
                else:
                    best_yolo_epoch = best_row + 1

    best_yolo_epoch = int(np.clip(best_yolo_epoch, 5, YOLO_EPOCHS))
    print("YOLO best weights:", yolo_best_path)
    print("Selected detector epoch (mask mAP proxy):", best_yolo_epoch)
else:
    USE_DETECTOR_FUSION = False
    print("Smoke mode: detector training skipped.")


In [ ]:
# ============================================================
# CELL 13 — TUNE DETECTOR-GUIDED FUSION ON TUNING VIEWS
# Select fusion settings on tuning observations only; report a separate holdout.
# ============================================================
if RUN_MODE == "competition":
    def predict_yolo_cached(detector, image_path, cache_dir):
        """Return scores + bit-packed full-resolution masks, cached by physical JPEG stem."""
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)
        image_path = Path(image_path)
        cache_path = cache_dir / f"{image_path.stem}.npz"

        if cache_path.is_file():
            with np.load(cache_path, allow_pickle=False) as saved:
                return (
                    saved["scores"].astype(np.float32, copy=False),
                    saved["packed_masks"],
                    tuple(int(x) for x in saved["shape"].tolist()),
                )

        with Image.open(image_path) as handle:
            original = np.asarray(handle.convert("L"))
        height, width = original.shape

        results = detector.predict(
            source=str(image_path),
            imgsz=1024,
            conf=0.001,
            iou=0.70,
            max_det=20,
            retina_masks=True,
            device=0,
            verbose=False,
        )
        result = results[0]
        if result.masks is None or result.boxes is None or len(result.boxes) == 0:
            scores = np.empty((0,), dtype=np.float32)
            masks = np.zeros((0, height, width), dtype=np.uint8)
        else:
            scores = result.boxes.conf.detach().float().cpu().numpy().astype(np.float32)
            masks = (result.masks.data.detach().float().cpu().numpy() >= 0.5).astype(np.uint8)
            if masks.shape[1:] != (height, width):
                masks = np.stack([
                    cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
                    for mask in masks
                ]).astype(np.uint8)
            order = np.argsort(scores)[::-1][:20]
            scores = scores[order]
            masks = masks[order]

        packed_masks = np.packbits(masks, axis=2)
        temporary = cache_path.with_suffix(".tmp.npz")
        np.savez_compressed(
            temporary,
            scores=scores.astype(np.float32),
            packed_masks=packed_masks,
            shape=np.asarray([height, width], dtype=np.int32),
        )
        temporary.replace(cache_path)
        return scores, packed_masks, (height, width)

    def unpack_yolo_prediction(packed_prediction):
        scores, packed_masks, shape = packed_prediction
        height, width = shape
        masks = np.unpackbits(
            packed_masks, axis=2, count=width
        ).astype(bool, copy=False)
        return scores, masks, shape

    def fuse_detector_unet(probability, detector_prediction, disk_mask, params):
        """Detector masks assign instance identity to U-Net filament pixels."""
        scores, detector_masks, shape = detector_prediction
        height, width = probability.shape
        binary = probability >= float(params["unet_threshold"])

        if bool(params.get("clip_to_disk", True)) and disk_mask is not None:
            binary &= disk_mask.astype(bool)

        close_radius = int(params.get("close_radius", 0))
        if close_radius > 0:
            kernel_size = close_radius * 2 + 1
            kernel = cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE, (kernel_size, kernel_size)
            )
            binary = cv2.morphologyEx(
                binary.astype(np.uint8), cv2.MORPH_CLOSE, kernel
            ).astype(bool)

        labels = np.zeros((height, width), dtype=np.int32)
        if detector_masks.shape[0] > 0:
            dilation_radius = int(params["dilation_radius"])
            if dilation_radius > 0:
                size = 2 * dilation_radius + 1
                kernel = cv2.getStructuringElement(
                    cv2.MORPH_ELLIPSE, (size, size)
                )
            else:
                kernel = None

            next_label = 1
            order = np.argsort(scores)[::-1]
            for j in order:
                if float(scores[j]) < float(params["score_threshold"]):
                    continue
                proposal = detector_masks[j]
                if proposal.shape != (height, width):
                    proposal = cv2.resize(
                        proposal.astype(np.uint8),
                        (width, height),
                        interpolation=cv2.INTER_NEAREST,
                    ).astype(bool)
                else:
                    proposal = proposal.astype(bool)

                if kernel is not None:
                    proposal = cv2.dilate(
                        proposal.astype(np.uint8), kernel
                    ).astype(bool)

                candidate = proposal & binary & (labels == 0)
                if int(candidate.sum()) < int(params["min_area"]):
                    continue
                labels[candidate] = next_label
                next_label += 1

        if bool(params["add_unclaimed_components"]):
            remaining = binary & (labels == 0)
            count, components = cv2.connectedComponents(
                remaining.astype(np.uint8), connectivity=8
            )
            next_label = int(labels.max()) + 1
            min_area = int(params["min_area"])
            for component_id in range(1, count):
                component = components == component_id
                if int(component.sum()) >= min_area:
                    labels[component] = next_label
                    next_label += 1

        return labels

    def pooled_instance_score(image_ids, labels_by_stem, index):
        pooled_tp = pooled_fp = pooled_fn = pooled_iou = 0.0
        one_to_many = many_to_one = 0
        dice_scores = []
        iou_scores = []
        image_pq = []
        for image_id in image_ids:
            meta = index.images[image_id]
            stem = Path(str(meta["file_name"])).stem
            predicted = labels_by_stem[stem]
            truth = build_instance_id_mask(index, image_id)
            metric = compute_pq(predicted, truth)
            semantic = compute_segmentation_metrics(predicted > 0, truth > 0)
            pooled_tp += float(metric["tp"])
            pooled_fp += float(metric["fp"])
            pooled_fn += float(metric["fn"])
            pooled_iou += float(metric["matched_iou_sum"])
            one_to_many += int(metric["one_to_many"])
            many_to_one += int(metric["many_to_one"])
            dice_scores.append(float(semantic["dice"]))
            iou_scores.append(float(semantic["iou"]))
            image_pq.append(float(metric["pq"]))

        denominator = pooled_tp + 0.5 * pooled_fp + 0.5 * pooled_fn
        return {
            "pq_pooled": pooled_iou / denominator if denominator else 1.0,
            "mean_image_pq": float(np.mean(image_pq)) if image_pq else 0.0,
            "mean_dice": float(np.mean(dice_scores)) if dice_scores else 0.0,
            "mean_iou": float(np.mean(iou_scores)) if iou_scores else 0.0,
            "tp": int(pooled_tp),
            "fp": int(pooled_fp),
            "fn": int(pooled_fn),
            "matched_iou_sum": pooled_iou,
            "one_to_many": one_to_many,
            "many_to_one": many_to_one,
            "annotation_views_scored": len(image_ids),
            "physical_images_scored": len({Path(index.images[i]["file_name"]).stem for i in image_ids}),
        }

    def score_unet(image_ids, unet_by_stem, postprocess_config):
        labels_by_stem = {}
        for image_id in image_ids:
            meta = index.images[image_id]
            stem = Path(str(meta["file_name"])).stem
            if stem not in labels_by_stem:
                probability, boundary, disk = unet_by_stem[stem]
                labels_by_stem[stem] = reconstruct_instances(
                    probability, postprocess_config, boundary, disk
                )
        return pooled_instance_score(image_ids, labels_by_stem, index)

    def score_fusion(image_ids, unet_by_stem, detections_by_stem, params):
        labels_by_stem = {}
        stems = sorted({Path(index.images[i]["file_name"]).stem for i in image_ids})
        for stem in stems:
            probability, boundary, disk = unet_by_stem[stem]
            prediction = detections_by_stem[stem]
            labels_by_stem[stem] = fuse_detector_unet(
                probability, prediction, disk, params
            )
        return pooled_instance_score(image_ids, labels_by_stem, index)

    unet_tune_by_stem = {}
    for image_id, entry in zip(tune_ids, tune_cache):
        stem = Path(index.images[image_id]["file_name"]).stem
        unet_tune_by_stem.setdefault(
            stem, (entry[0], entry[1], entry[3])
        )

    unet_fusion_holdout_by_stem = {}
    for image_id, entry in zip(fusion_holdout_ids, fusion_holdout_cache):
        stem = Path(index.images[image_id]["file_name"]).stem
        unet_fusion_holdout_by_stem.setdefault(
            stem, (entry[0], entry[1], entry[3])
        )

    # Unpack each image's bit-packed proposals once; this keeps the search fast
    # without retaining floating-point mask logits for every detector instance.
    image_path_by_stem = {
        Path(meta["file_name"]).stem: data.train_images / meta["file_name"]
        for meta in index.images.values()
    }
    yolo_cache_dir = WORKING / f"yolo_prediction_cache_{RUN_TAG}"
    yolo_tune_predictions = {
        stem: unpack_yolo_prediction(
            predict_yolo_cached(yolo_model, image_path_by_stem[stem], yolo_cache_dir)
        )
        for stem in sorted(tune_stems)
    }
    yolo_holdout_predictions = {
        stem: unpack_yolo_prediction(
            predict_yolo_cached(yolo_model, image_path_by_stem[stem], yolo_cache_dir)
        )
        for stem in fusion_holdout_stems
    }

    tune_unet_metrics = score_unet(tune_ids, unet_tune_by_stem, optimized_config)
    fusion_grid = []
    for score_threshold, unet_threshold, dilation_radius, min_area, add_unclaimed in itertools.product(
        (0.10, 0.25),
        (0.30, 0.45, 0.60),
        (2, 5),
        (100, 300),
        (False, True),
    ):
        params = {
            "score_threshold": float(score_threshold),
            "unet_threshold": float(unet_threshold),
            "dilation_radius": int(dilation_radius),
            "min_area": int(min_area),
            "add_unclaimed_components": bool(add_unclaimed),
            "close_radius": 2,
            "clip_to_disk": bool(optimized_config.use_solar_disk_mask),
        }
        metrics = score_fusion(
            tune_ids, unet_tune_by_stem, yolo_tune_predictions, params
        )
        fusion_grid.append({**params, **metrics})

    fusion_search_table = pd.DataFrame(fusion_grid).sort_values(
        ["pq_pooled", "mean_dice", "mean_image_pq"], ascending=False
    ).reset_index(drop=True)

    best_row = fusion_search_table.iloc[0].to_dict()
    fusion_best_params = {
        "score_threshold": float(best_row["score_threshold"]),
        "unet_threshold": float(best_row["unet_threshold"]),
        "dilation_radius": int(best_row["dilation_radius"]),
        "min_area": int(best_row["min_area"]),
        "add_unclaimed_components": bool(best_row["add_unclaimed_components"]),
        "close_radius": int(best_row["close_radius"]),
        "clip_to_disk": bool(best_row["clip_to_disk"]),
    }
    fusion_best_tune_metrics = {
        key: best_row[key] for key in (
            "pq_pooled", "mean_image_pq", "mean_dice", "mean_iou",
            "tp", "fp", "fn", "one_to_many", "many_to_one",
        )
    }
    baseline_gain = float(fusion_best_tune_metrics["pq_pooled"]) - float(tune_unet_metrics["pq_pooled"])
    USE_DETECTOR_FUSION = baseline_gain > 0.001

    # Freeze the method/configuration using tuning observations only.
    fusion_holdout_metrics = score_fusion(
        fusion_holdout_ids,
        unet_fusion_holdout_by_stem,
        yolo_holdout_predictions,
        fusion_best_params,
    )
    fusion_holdout_unet_metrics = score_unet(
        fusion_holdout_ids,
        unet_fusion_holdout_by_stem,
        optimized_config,
    )

    fusion_search_path = WORKING / f"yolo_fusion_search_{RUN_TAG}.csv"
    fusion_search_table.to_csv(fusion_search_path, index=False)
    with (WORKING / f"yolo_fusion_config_{RUN_TAG}.json").open("w") as handle:
        json.dump(fusion_best_params, handle, indent=2)
    with (WORKING / f"yolo_fusion_metrics_{RUN_TAG}.json").open("w") as handle:
        json.dump({
            "tuning_unet_baseline": tune_unet_metrics,
            "tuning_best_fusion": fusion_best_tune_metrics,
            "tuning_pooled_pq_gain": baseline_gain,
            "selected_detector_fusion": USE_DETECTOR_FUSION,
            "independent_fusion_holdout": fusion_holdout_metrics,
            "independent_holdout_unet_baseline": fusion_holdout_unet_metrics,
        }, handle, indent=2)

    print("U-Net tune baseline:", json.dumps(tune_unet_metrics, indent=2))
    print("Best fusion tune metrics:", json.dumps(fusion_best_tune_metrics, indent=2))
    print("Tune pooled-PQ gain:", baseline_gain)
    print("Selected detector fusion:", USE_DETECTOR_FUSION)
    print("Independent fusion holdout:", json.dumps(fusion_holdout_metrics, indent=2))
    print("Independent U-Net holdout (same views):", json.dumps(fusion_holdout_unet_metrics, indent=2))
    display(fusion_search_table.head(12))

    # Free large model prediction maps before final all-data training.
    del tune_cache, fusion_holdout_cache
    del unet_tune_by_stem, unet_fusion_holdout_by_stem
    del yolo_tune_predictions, yolo_holdout_predictions
    del yolo_model
    gc.collect()
    torch.cuda.empty_cache()
else:
    USE_DETECTOR_FUSION = False
    fusion_best_params = None
    fusion_holdout_metrics = {"note": "smoke mode does not train the detector"}
    print("Smoke mode: detector fusion skipped.")


In [ ]:
# ============================================================
# CELL 12 — RETRAIN FINAL MODEL ON ALL LABELLED ANNOTATION VIEWS
# Hyperparameters were chosen before using the held-out metrics.
# ============================================================
if RUN_MODE == "competition" and FINAL_RETRAIN_ALL_DATA:
    # Free the validation model, dataloaders, and cached probability maps before
    # allocating the all-data model. This helps avoid GPU/RAM exhaustion.
    for _name in (
        "model", "optimizer", "scheduler", "scaler", "train_loader",
        "train_dataset", "quick_val_dataset", "tune_dataset", "holdout_dataset",
        "tune_cache", "fusion_holdout_dataset", "fusion_holdout_cache",
    ):
        globals().pop(_name, None)
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    print("Building all-data training dataset...")
    all_train_ids = sorted(index.images)
    final_config = PipelineConfig(**{
        **asdict(optimized_config),
        "epochs": best_epoch,
        "samples_per_epoch": config.samples_per_epoch,
        "max_train_images": None,
    })

    final_dataset = SolarFilamentDataset(
        index, data.train_images, all_train_ids, final_config, training=True
    )
    final_loader = DataLoader(
        final_dataset,
        batch_size=final_config.batch_size,
        shuffle=True,
        num_workers=final_config.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(final_config.num_workers > 0),
    )

    final_model = FilamentSegmentationModel(final_config).to(device)
    final_optimizer = torch.optim.AdamW(
        final_model.parameters(),
        lr=final_config.learning_rate,
        weight_decay=final_config.weight_decay,
    )
    final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        final_optimizer, T_max=max(1, final_config.epochs)
    )
    final_scaler = build_grad_scaler(device, final_config.use_amp)
    final_training_history = []

    for epoch in range(final_config.epochs):
        epoch_started = time.perf_counter()
        loss = train_one_epoch(
            final_model,
            final_loader,
            final_optimizer,
            final_scaler,
            final_config,
            device,
            pos_weight,
        )
        final_scheduler.step()
        row = {
            "epoch": epoch + 1,
            "epochs": final_config.epochs,
            "train_loss": loss,
            "seconds": time.perf_counter() - epoch_started,
        }
        final_training_history.append(row)
        print(row)

    final_model_path = WORKING / f"final_model_all_data_{RUN_TAG}.pth"
    torch.save({
        "model": final_model.state_dict(),
        "config": asdict(final_config),
        "trained_on_all_annotation_views": True,
        "selected_epoch_count": best_epoch,
        "repo_commit": REPO_COMMIT,
    }, final_model_path)
    pd.DataFrame(final_training_history).to_csv(
        WORKING / f"final_training_history_{RUN_TAG}.csv",
        index=False,
    )
    model_for_submission = final_model
    submission_config = final_config
    del final_loader, final_dataset, final_optimizer, final_scheduler, final_scaler
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("Saved all-data checkpoint:", final_model_path)
else:
    model_for_submission = model.to(device)
    submission_config = optimized_config
    final_model_path = best_path
    print("Using best validation-selected model; no all-data retrain requested.")


In [ ]:
# ============================================================
# CELL 15 — OPTIONAL ALL-DATA YOLO FINE-TUNING
# Only run this if detector fusion beat the U-Net baseline on tuning data.
# ============================================================
final_yolo_weights = None

if RUN_MODE == "competition" and USE_DETECTOR_FUSION:
    if FINAL_RETRAIN_ALL_DATA:
        # Move the final U-Net to CPU temporarily so the YOLO training fits GPU memory.
        model_for_submission.to("cpu")
        gc.collect()
        torch.cuda.empty_cache()

        all_view_ids = sorted(index.images)
        all_yolo_stats = export_yolo_views(all_view_ids, "all_train")
        all_yaml = YOLO_DATA_ROOT / "dataset_all.yaml"
        all_yaml.write_text(
            f"path: {YOLO_DATA_ROOT}\n"
            "train: images/all_train\n"
            "val: images/all_train\n"
            "names:\n"
            "  0: filament\n",
            encoding="utf-8",
        )
        print("All-data YOLO export:", json.dumps(all_yolo_stats, indent=2))

        final_yolo_run_name = f"all_data_{RUN_TAG}"
        final_yolo_model = YOLO(str(yolo_best_path))
        final_yolo_epochs = max(5, min(10, int(np.ceil(best_yolo_epoch / 3))))
        final_yolo_model.train(
            data=str(all_yaml),
            imgsz=1024,
            epochs=final_yolo_epochs,
            batch=2,
            device=0,
            workers=2,
            patience=0,
            seed=SEED,
            deterministic=False,
            single_cls=True,
            cos_lr=True,
            plots=False,
            cache=False,
            val=False,
            mosaic=0.0,
            close_mosaic=0,
            mixup=0.0,
            copy_paste=0.0,
            fliplr=0.5,
            flipud=0.5,
            degrees=0.0,
            translate=0.03,
            scale=0.10,
            hsv_h=0.0,
            hsv_s=0.0,
            hsv_v=0.10,
            overlap_mask=True,
            mask_ratio=2,
            project=str(YOLO_RUN_ROOT),
            name=final_yolo_run_name,
            exist_ok=True,
        )
        final_yolo_weights = (
            YOLO_RUN_ROOT / final_yolo_run_name / "weights" / "last.pt"
        )
        if not final_yolo_weights.is_file():
            raise FileNotFoundError(
                f"All-data YOLO fine-tuning did not produce last.pt: {final_yolo_weights}"
            )
        del final_yolo_model
        gc.collect()
        torch.cuda.empty_cache()
        model_for_submission.to(device)
        print("Final all-data YOLO weights:", final_yolo_weights)
    else:
        final_yolo_weights = yolo_best_path
        print("Using split-trained YOLO weights:", final_yolo_weights)
else:
    print("Detector fusion was not selected; final submission will use the tuned U-Net pipeline.")


In [ ]:
# ============================================================
# CELL 13 — PREDICT ALL TEST IMAGES + VALIDATE SUBMISSION CSV
# ============================================================
if RUN_MODE == "competition":
    submission_path = WORKING / "submission.csv"

    if USE_DETECTOR_FUSION:
        final_detector = YOLO(str(final_yolo_weights))
        # YOLO predict receives device=0 explicitly; the wrapper need not implement .to().
        test_prediction_cache = WORKING / f"test_yolo_cache_{RUN_TAG}"
        submission_rows = []
        per_test_counts = []

        for image_path in list_image_files(data.test_images):
            with Image.open(image_path) as handle:
                image = np.asarray(handle.convert("L"))

            probability, boundary = predict_tiled(
                model_for_submission, image, submission_config, device
            )
            disk = detect_solar_disk(image)
            packed_prediction = predict_yolo_cached(
                final_detector, image_path, test_prediction_cache
            )
            detector_prediction = unpack_yolo_prediction(packed_prediction)

            labels = fuse_detector_unet(
                probability,
                detector_prediction,
                disk,
                {
                    **fusion_best_params,
                    "clip_to_disk": bool(submission_config.use_solar_disk_mask),
                },
            )

            count = 0
            for label in np.unique(labels):
                if int(label) == 0:
                    continue
                instance_mask = labels == int(label)
                if not instance_mask.any():
                    continue
                validate_rle(instance_mask)
                submission_rows.append({
                    "filament_id": f"{image_path.stem}_{int(label)}",
                    "segmentation_rle": mask_to_rle(instance_mask),
                })
                count += 1
            per_test_counts.append(count)

        submission = pd.DataFrame(
            submission_rows,
            columns=["filament_id", "segmentation_rle"],
        )
        submission.to_csv(submission_path, index=False)
        del final_detector
        gc.collect()
        torch.cuda.empty_cache()
    else:
        submission = create_submission(
            model=model_for_submission,
            data=data,
            config=submission_config,
            output_path=submission_path,
            device=device,
        )

    assert list(submission.columns) == ["filament_id", "segmentation_rle"]
    assert submission["filament_id"].is_unique
    assert not submission.isna().any().any()
    assert len(submission) > 0

    test_paths = sorted(
        p for p in Path(data.test_images).iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    )
    test_stems = {p.stem for p in test_paths}
    output_stems = {
        str(fid).rsplit("_", 1)[0]
        for fid in submission["filament_id"].tolist()
    }
    assert output_stems <= test_stems, "Submission contains unknown observation IDs."

    print("SUBMISSION CREATED:", submission_path)
    print("Detector fusion selected:", USE_DETECTOR_FUSION)
    print("Test observations:", len(test_paths))
    print("Submission rows:", len(submission))
    if USE_DETECTOR_FUSION:
        print("Instances per test image (min/median/max):", (
            int(np.min(per_test_counts)),
            float(np.median(per_test_counts)),
            int(np.max(per_test_counts)),
        ))
    print("Predicted instances per test observation:", round(len(submission) / max(1, len(test_paths)), 2))
    display(submission.head(10))
else:
    print("SMOKE MODE: test submission disabled intentionally.")


In [ ]:
# ============================================================
# CELL 14 — REPRODUCIBILITY SUMMARY
# ============================================================
summary = {
    "run_tag": RUN_TAG,
    "repo_branch": REPO_BRANCH,
    "repo_commit": REPO_COMMIT,
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision_available": TORCHVISION_OK,
    "torchvision_error": TORCHVISION_ERROR,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "run_mode": RUN_MODE,
    "best_epoch": best_epoch,
    "best_quick_global_pq": best_metric,
    "independent_holdout_metrics": holdout_metrics,
    "holdout_eval_seconds": holdout_seconds,
    "num_train_annotation_views": len(train_ids),
    "num_val_annotation_views": len(val_ids),
    "num_train_physical_images": len(physical_train),
    "num_val_physical_images": len(physical_val),
    "final_checkpoint": str(final_model_path),
    "detector_fusion_selected_on_tuning": bool(globals().get("USE_DETECTOR_FUSION", False)),
    "fusion_parameters": globals().get("fusion_best_params"),
    "fusion_holdout_metrics": globals().get("fusion_holdout_metrics"),
    "detector_best_epoch": globals().get("best_yolo_epoch"),
    "detector_final_weights": str(globals().get("final_yolo_weights")) if globals().get("final_yolo_weights") else None,
    "submission_path": str(WORKING / "submission.csv") if RUN_MODE == "competition" else None,
}

summary_path = WORKING / f"run_summary_{RUN_TAG}.json"
summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print(json.dumps(summary, indent=2, default=str))
print("\nArtifacts in /kaggle/working can be downloaded from the notebook Output panel.")
